# 02 Lead History Assign Sub ID

This notebook assigns subsystem ids to the cleaned new data.

Rules:
- preserve original `system_name`
- if a `system_name` has already appeared under the same base PWSID in history, reuse the existing sub-id
- if a `system_name` is new, assign the next available letter
- always output review files for human double check

Enter the input file names in the next cell.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('.').resolve()
ASSIGN_OUTPUT_DIR = BASE_DIR / 'assign_pipeline'
ASSIGN_OUTPUT_DIR.mkdir(exist_ok=True)

history_file_name = input(
    'Enter the history file name in the current folder '
    '(example: lead_90th_history_no_duplicate_renamed.csv): '
).strip()

if not history_file_name:
    raise ValueError('You must enter a history file name.')

HISTORY_FILE_PATH = BASE_DIR / history_file_name

NEW_REVIEWED_FILE_PATH = (
    BASE_DIR / 'clean_pipeline' / 'Public_Water_Supply_90th_Percentiles_2025_cleaned_reviewed.csv'
)

if not HISTORY_FILE_PATH.exists():
    raise FileNotFoundError(f'History file not found: {HISTORY_FILE_PATH}')

if not NEW_REVIEWED_FILE_PATH.exists():
    raise FileNotFoundError(f'New reviewed file not found: {NEW_REVIEWED_FILE_PATH}')

print('History file:', HISTORY_FILE_PATH)
print('New reviewed file:', NEW_REVIEWED_FILE_PATH)
print('Output folder:', ASSIGN_OUTPUT_DIR)

History file: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\lead_90th_history_no_duplicate_renamed.csv
New reviewed file: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\clean_pipeline\Public_Water_Supply_90th_Percentiles_2025_cleaned_reviewed.csv
Output folder: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\assign_pipeline


In [2]:
STANDARD_COLUMNS = [
    'pwsid',
    'system_name',
    'county',
    'population',
    'monitoring_end_date',
    'lead_90th_ppb',
    'includes_5th_liter_or_not',
    'sampling_next_due_subject_to_change',
    'year',
    'month',
    'above_action_level',
]


def load_clean_csv(path):
    df = pd.read_csv(path)
    missing = [col for col in STANDARD_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'Missing expected columns in {path.name}: {missing}')

    df = df[STANDARD_COLUMNS].copy()
    df['monitoring_end_date'] = pd.to_datetime(df['monitoring_end_date'], errors='coerce')
    df['sampling_next_due_subject_to_change'] = pd.to_datetime(
        df['sampling_next_due_subject_to_change'],
        errors='coerce',
    )

    for col in ['pwsid', 'system_name', 'county', 'includes_5th_liter_or_not']:
        df[col] = df[col].astype(str).str.strip()

    return df


def extract_base_pwsid(value):
    value = str(value).strip()
    if '-' in value:
        prefix, suffix = value.rsplit('-', 1)
        if suffix.isalpha():
            return prefix
    return value


def build_history_mapping(history_df):
    mapping = {}
    used_letters = {}

    for row in history_df[['pwsid', 'system_name']].drop_duplicates().itertuples(index=False):
        existing_pwsid = row.pwsid
        system_name = row.system_name
        base_pwsid = extract_base_pwsid(existing_pwsid)

        mapping.setdefault(base_pwsid, {})[system_name] = existing_pwsid
        used_letters.setdefault(base_pwsid, set())

        suffix = str(existing_pwsid).replace(base_pwsid, '', 1).lstrip('-')
        if suffix.isalpha():
            used_letters[base_pwsid].add(suffix)

    return mapping, used_letters


def next_available_suffix(used_suffixes):
    for letter in 'abcdefghijklmnopqrstuvwxyz':
        if letter not in used_suffixes:
            return letter
    raise ValueError('Ran out of suffix letters')


def assign_subsystem_pwsid(new_df, history_mapping, used_letters):
    assigned = new_df.copy()
    mapping_rows = []
    assigned_rows = []

    for row in assigned.itertuples(index=False):
        base_pwsid = extract_base_pwsid(row.pwsid)
        system_name = row.system_name

        history_mapping.setdefault(base_pwsid, {})
        used_letters.setdefault(base_pwsid, set())

        if system_name in history_mapping[base_pwsid]:
            resolved_pwsid = history_mapping[base_pwsid][system_name]
            assignment_status = 'reused_existing'
        else:
            suffix = next_available_suffix(used_letters[base_pwsid])
            resolved_pwsid = f'{base_pwsid}-{suffix}'
            history_mapping[base_pwsid][system_name] = resolved_pwsid
            used_letters[base_pwsid].add(suffix)
            assignment_status = 'assigned_new'

        assigned_rows.append(resolved_pwsid)
        mapping_rows.append(
            {
                'base_pwsid': base_pwsid,
                'system_name': system_name,
                'resolved_pwsid': resolved_pwsid,
                'assignment_status': assignment_status,
            }
        )

    assigned['pwsid'] = assigned_rows
    mapping_df = (
        pd.DataFrame(mapping_rows)
        .drop_duplicates()
        .sort_values(['base_pwsid', 'resolved_pwsid'])
        .reset_index(drop=True)
    )

    assignment_review_df = mapping_df[mapping_df['assignment_status'] == 'assigned_new'].reset_index(drop=True)
    assigned = assigned.sort_values(['pwsid', 'monitoring_end_date', 'system_name']).reset_index(drop=True)

    return assigned, mapping_df, assignment_review_df


In [3]:
history_df = load_clean_csv(HISTORY_FILE_PATH)
new_reviewed_df = load_clean_csv(NEW_REVIEWED_FILE_PATH)

history_mapping, used_letters = build_history_mapping(history_df)
assigned_new_df, mapping_df, assignment_review_df = assign_subsystem_pwsid(
    new_reviewed_df,
    history_mapping,
    used_letters,
)

summary_df = pd.DataFrame([
    {
        'history_rows': len(history_df),
        'new_reviewed_rows': len(new_reviewed_df),
        'assigned_rows': len(assigned_new_df),
        'new_assignments_to_review': len(assignment_review_df),
    }
])

display(summary_df)
display(mapping_df.head())
display(assignment_review_df.head())
display(assigned_new_df.head())

,history_rows,new_reviewed_rows,assigned_rows,new_assignments_to_review
0,6043,1385,1385,65


,base_pwsid,system_name,resolved_pwsid,assignment_status
0,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,MI0000011,reused_existing
1,MI0000012,ADA TOWNSHIP,MI0000012,reused_existing
2,MI0000020,ADAMS TOWNSHIP,MI0000020,reused_existing
3,MI0000030,ADDISON,MI0000030,reused_existing
4,MI0000040,ADRIAN,MI0000040,reused_existing


,base_pwsid,system_name,resolved_pwsid,assignment_status
0,MI0000324,AUSABLE VALLEY COMMUNITY (DS002),MI0000324-e,assigned_new
1,MI0000324,AUSABLE VALLEY COMMUNITY (DS001),MI0000324-f,assigned_new
2,MI0000324,AUSABLE VALLEY COMMUNITY (DS003),MI0000324-g,assigned_new
3,MI0000347,BAIR LAKE BIBLE CAMP (DS003),MI0000347-l,assigned_new
4,MI0000347,BAIR LAKE BIBLE CAMP (DS004),MI0000347-m,assigned_new


,pwsid,system_name,county,population,monitoring_end_date,lead_90th_ppb,includes_5th_liter_or_not,sampling_next_due_subject_to_change,year,month,above_action_level
0,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,GRAND TRAVERSE,128,2023-12-31,0,N,2026-09-30,2023,12,False
1,MI0000012,ADA TOWNSHIP,KENT,7584,2025-12-31,0,N,2026-09-30,2025,12,False
2,MI0000020,ADAMS TOWNSHIP,HOUGHTON,2010,2025-12-31,0,N,2028-09-30,2025,12,False
3,MI0000030,ADDISON,LENAWEE,605,2023-12-31,2,N,2026-09-30,2023,12,False
4,MI0000040,ADRIAN,LENAWEE,23663,2023-12-31,0,N,2026-09-30,2023,12,False


In [4]:
output_stem = Path(NEW_REVIEWED_FILE_PATH).stem.replace('_cleaned_reviewed', '')

assigned_path = ASSIGN_OUTPUT_DIR / f'{output_stem}_with_assigned_pwsid.csv'
mapping_path = ASSIGN_OUTPUT_DIR / f'{output_stem}_proposed_mapping.csv'
assignment_review_path = ASSIGN_OUTPUT_DIR / f'{output_stem}_assignment_review.csv'

assigned_new_df.to_csv(assigned_path, index=False, date_format='%Y-%m-%d')
mapping_df.to_csv(mapping_path, index=False)
assignment_review_df.to_csv(assignment_review_path, index=False)

print('Saved:', assigned_path)
print('Saved:', mapping_path)
print('Saved:', assignment_review_path)

Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\assign_pipeline\Public_Water_Supply_90th_Percentiles_2025_with_assigned_pwsid.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\assign_pipeline\Public_Water_Supply_90th_Percentiles_2025_proposed_mapping.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\assign_pipeline\Public_Water_Supply_90th_Percentiles_2025_assignment_review.csv
